# <center> Deep Learning Coursework </center>
## <center> Audio Text Transformer Implementation </center>

Explaination of Audio Text Transformer

Imports required

In [6]:
from pathlib import Path
import torch
import torch.backends.cudnn
import numpy as np
from torch import nn, optim
from torch.nn import functional as F
import torchvision.datasets
from torch.optim.optimizer import Optimizer
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision import transforms
from torch.utils.tensorboard import SummaryWriter

from audio_text_transformer import (
    DEVICE,
    AudioTextDataset,
    AudioTextTransformer,
    Trainer,
    get_summary_writer_log_dir,
    load_features,
    set_seed,
)

DATASET_ROOT = Path("/data/COMS30017")
LOG_DIR = Path("logs")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


load data centres

In [7]:
set_seed(42)

train_audio, train_text, train_labels = load_features(DATASET_ROOT, "train")
val_audio, val_text, val_labels = load_features(DATASET_ROOT, "val")

train_dataset = AudioTextDataset(train_audio, train_text, train_labels, training=True)
val_dataset = AudioTextDataset(val_audio, val_text, val_labels, training=False)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=0)

print("Train samples:", len(train_dataset), "| Val samples:", len(val_dataset))

Train samples: 24468 | Val samples: 750


baseline mean pooling

In [8]:
set_seed(42)
sample_a, sample_t, _ = next(iter(train_loader))

mean_model = AudioTextTransformer(
    audio_dim=sample_a.shape[-1],
    text_dim=sample_t.shape[-1],
    pooling="mean",
)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mean_model.parameters(), lr=1e-3)

log_dir = get_summary_writer_log_dir(
    type("Args", (), {"log_dir": Path("logs"), "batch_size": 128, "learning_rate": 1e-3})()
)
writer = SummaryWriter(str(log_dir), flush_secs=5)

mean_trainer = Trainer(mean_model, train_loader, val_loader, criterion, optimizer, writer, DEVICE)
mean_trainer.train(epochs=10, val_frequency=2, print_frequency=10, log_frequency=10)
torch.save(mean_trainer.model.state_dict(), "weights_mean.pth")
writer.close()

Number of trainable parameters: 955,266
epoch: [0], step: [9/192], batch loss: 0.69716, batch accuracy: 52.34, data load time: 0.00288, step time: 0.01115
epoch: [0], step: [19/192], batch loss: 0.69343, batch accuracy: 51.56, data load time: 0.00274, step time: 0.01026
epoch: [0], step: [29/192], batch loss: 0.69290, batch accuracy: 48.44, data load time: 0.00279, step time: 0.00995
epoch: [0], step: [39/192], batch loss: 0.69935, batch accuracy: 51.56, data load time: 0.00296, step time: 0.00990
epoch: [0], step: [49/192], batch loss: 0.70856, batch accuracy: 46.88, data load time: 0.00290, step time: 0.00995
epoch: [0], step: [59/192], batch loss: 0.69476, batch accuracy: 53.12, data load time: 0.00270, step time: 0.01003
epoch: [0], step: [69/192], batch loss: 0.70947, batch accuracy: 47.66, data load time: 0.00284, step time: 0.01005
epoch: [0], step: [79/192], batch loss: 0.69735, batch accuracy: 49.22, data load time: 0.00294, step time: 0.00992
epoch: [0], step: [89/192], batch

extension

In [10]:
set_seed(42)
sample_a, sample_t, _ = next(iter(train_loader))

attn_model = AudioTextTransformer(
    audio_dim=sample_a.shape[-1],
    text_dim=sample_t.shape[-1],
    pooling="attention",
)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(attn_model.parameters(), lr=1e-3)

log_dir = get_summary_writer_log_dir(
    type("Args", (), {"log_dir": Path("logs"), "batch_size": 128, "learning_rate": 1e-3})()
)
writer = SummaryWriter(str(log_dir), flush_secs=5)

attn_trainer = Trainer(attn_model, train_loader, val_loader, criterion, optimizer, writer, DEVICE)
attn_trainer.train(epochs=10, val_frequency=2, print_frequency=10, log_frequency=10)
torch.save(attn_trainer.model.state_dict(), "weights_attention.pth")
writer.close()

Number of trainable parameters: 988,290
epoch: [0], step: [9/192], batch loss: 0.70284, batch accuracy: 46.09, data load time: 0.00319, step time: 0.01163
epoch: [0], step: [19/192], batch loss: 0.69258, batch accuracy: 52.34, data load time: 0.00315, step time: 0.01113
epoch: [0], step: [29/192], batch loss: 0.69267, batch accuracy: 48.44, data load time: 0.00326, step time: 0.01177
epoch: [0], step: [39/192], batch loss: 0.69449, batch accuracy: 49.22, data load time: 0.00345, step time: 0.01082
epoch: [0], step: [49/192], batch loss: 0.69201, batch accuracy: 52.34, data load time: 0.00326, step time: 0.01089
epoch: [0], step: [59/192], batch loss: 0.69624, batch accuracy: 50.00, data load time: 0.00330, step time: 0.01078
epoch: [0], step: [69/192], batch loss: 0.69716, batch accuracy: 50.78, data load time: 0.00320, step time: 0.01094
epoch: [0], step: [79/192], batch loss: 0.69539, batch accuracy: 46.88, data load time: 0.00332, step time: 0.01080
epoch: [0], step: [89/192], batch

compare accuracy

In [11]:
def evaluate_checkpoint(weights_path: Path, pooling: str, loader: DataLoader):
    sample_a, sample_t, _ = next(iter(loader))
    eval_model = AudioTextTransformer(
        audio_dim=sample_a.shape[-1],
        text_dim=sample_t.shape[-1],
        pooling=pooling,
    ).to(DEVICE)

    state_dict = torch.load(weights_path, map_location=DEVICE, weights_only=True)
    eval_model.load_state_dict(state_dict)
    eval_model.eval()

    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    preds_all, labels_all = [], []

    with torch.no_grad():
        for audio, text, labels in loader:
            audio = audio.to(DEVICE)
            text = text.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = eval_model(audio, text)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds_all.extend(logits.argmax(dim=-1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    acc = float(np.mean(np.array(preds_all) == np.array(labels_all)))
    avg_loss = total_loss / len(loader)
    return eval_model, avg_loss, acc


_, mean_loss, mean_acc = evaluate_checkpoint(Path("weights_mean.pth"), "mean", val_loader)
attn_model, attn_loss, attn_acc = evaluate_checkpoint(Path("weights_attention.pth"), "attention", val_loader)

print(f"Mean Pooling      -> Val Loss: {mean_loss:.5f} | Val Acc: {mean_acc * 100:.2f}%")
print(f"Attention Pooling -> Val Loss: {attn_loss:.5f} | Val Acc: {attn_acc * 100:.2f}%")

Mean Pooling      -> Val Loss: 0.69316 | Val Acc: 49.87%
Attention Pooling -> Val Loss: 0.69336 | Val Acc: 49.87%


In [12]:
for name, tensor in torch.load("weights_attention.pth", map_location="cpu", weights_only=True).items():
    print(f"{name:35s} | shape: {str(tuple(tensor.shape)):15s} | mean: {tensor.float().mean():+.4f}")

audio_projection.0.weight           | shape: (256, 768)      | mean: +0.0001
audio_projection.0.bias             | shape: (256,)          | mean: -0.0003
audio_projection.2.weight           | shape: (256,)          | mean: +0.9993
audio_projection.2.bias             | shape: (256,)          | mean: +0.0004
text_projection.0.weight            | shape: (256, 768)      | mean: -0.0001
text_projection.0.bias              | shape: (256,)          | mean: -0.0007
text_projection.2.weight            | shape: (256,)          | mean: +0.9986
text_projection.2.bias              | shape: (256,)          | mean: +0.0003
transformer.self_attn.in_proj_weight | shape: (768, 256)      | mean: +0.0001
transformer.self_attn.in_proj_bias  | shape: (768,)          | mean: -0.0001
transformer.self_attn.out_proj.weight | shape: (256, 256)      | mean: -0.0003
transformer.self_attn.out_proj.bias | shape: (256,)          | mean: +0.0003
transformer.linear1.weight          | shape: (512, 256)      | mean: +0.0

In [13]:
%run audio-text_transformer.py --learning-rate 1e-4 --seed 42

Training dataset size: 24468
Training batches: 192
Audio batch shape: torch.Size([128, 768])
Text batch shape: torch.Size([128, 40, 768])
Labels shape: torch.Size([128])
Writing logs to logs/AudioText_bs=128_lr=0.0001_run_0
Number of trainable parameters: 955,266
epoch: [0], step: [9/192], batch loss: 0.69426, batch accuracy: 50.78, data load time: 0.00672, step time: 0.01113
epoch: [0], step: [19/192], batch loss: 0.70598, batch accuracy: 48.44, data load time: 0.00391, step time: 0.01047
epoch: [0], step: [29/192], batch loss: 0.69471, batch accuracy: 53.12, data load time: 0.00370, step time: 0.00981
epoch: [0], step: [39/192], batch loss: 0.70740, batch accuracy: 51.56, data load time: 0.00363, step time: 0.00998
epoch: [0], step: [49/192], batch loss: 0.69920, batch accuracy: 48.44, data load time: 0.00603, step time: 0.00998
epoch: [0], step: [59/192], batch loss: 0.70468, batch accuracy: 50.78, data load time: 0.00382, step time: 0.01002
epoch: [0], step: [69/192], batch loss: 0

In [14]:
weights = torch.load(
    "transformer_weights.pth",
    map_location="cpu",
    weights_only=True
)

for name, tensor in weights.items():
    print(f"Layer: {name}")
    print(f"Shape: {tensor.shape}")
    print(f"Weights:\n{tensor}")
    print("-" * 50)

Layer: audio_projection.0.weight
Shape: torch.Size([256, 768])
Weights:
tensor([[-0.0295, -0.0165,  0.0188,  ..., -0.0004, -0.0362, -0.0224],
        [-0.0322, -0.0063, -0.0148,  ...,  0.0048,  0.0262,  0.0047],
        [ 0.0177, -0.0194, -0.0167,  ..., -0.0415,  0.0005, -0.0210],
        ...,
        [-0.0369,  0.0112, -0.0223,  ..., -0.0035,  0.0314,  0.0356],
        [ 0.0007,  0.0286, -0.0305,  ..., -0.0210,  0.0116, -0.0227],
        [-0.0020, -0.0204,  0.0346,  ..., -0.0044,  0.0195, -0.0036]])
--------------------------------------------------
Layer: audio_projection.0.bias
Shape: torch.Size([256])
Weights:
tensor([ 1.0117e-02,  2.7697e-02, -1.8717e-02,  1.7400e-02, -1.2324e-02,
         1.3006e-02, -5.0575e-03,  3.0910e-02,  2.3573e-02,  2.4645e-02,
        -3.2430e-02, -3.3816e-02, -1.4568e-02, -7.9001e-03,  8.8934e-03,
        -1.4703e-02,  1.5655e-02, -2.3021e-02,  1.6824e-02, -1.9640e-02,
        -2.8430e-02,  3.5594e-03,  1.3512e-02,  1.1618e-02,  2.5605e-02,
        -1.50

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torch import nn

model = attn_model  # or: model = mean_model
model.to(DEVICE)

dataset_root = Path("/data/COMS30017")

# Load validation features
val_audio, val_text, val_labels = load_features(
    dataset_root, "val"
)

# Create validation dataset
val_dataset = AudioTextDataset(
    val_audio, val_text, val_labels, training=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

# Evaluation setup
criterion = nn.CrossEntropyLoss()

model.eval()

total_loss = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for audio, text, labels in val_loader:
        audio = audio.to(DEVICE)
        text = text.to(DEVICE)
        labels = labels.to(DEVICE)

        # Get model predictions
        logits = model(audio, text)

        # Calculate loss
        loss = criterion(logits, labels)
        total_loss += loss.item()

        # Get predicted classes
        predictions = logits.argmax(dim=-1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate metrics
accuracy = np.mean(
    np.array(all_preds) == np.array(all_labels)
)

average_loss = total_loss / len(val_loader)

print("========== MODEL EVALUATION ==========")
print(f"Validation Loss: {average_loss:.5f}")
print(f"Validation Accuracy: {accuracy * 100:.2f}%")
print(f"Total Samples: {len(all_labels)}")
print("======================================")
